# Full pipeline for preprosessing, labeling and segmenting a test file set, for 6 sensors
## Outdated: the full preprocessing pipeline is instead in run_full_preprocessing_pipeline.py 
## Labeling and repetion segmentation was still done here

The M.Sc. spring 2026 edition, though this is heavily based on the code from the specialization project conducted in the fall of 2026, which again is heavily derived off of Maria's masters thesis: https://nva.sikt.no/registration/019aaa5d3e02-1f393658-c1a9-4f74-96a3-c428dc10c170

### Imports

In [ ]:

from pre_processing.labeling import detect_spikes, init_label, plot_signal_peaks, extract_activity_windows, apply_corrected_labels, extract_pressure_data, remove_idle, sort_push_pull
from pre_processing.fiks_time_muse import make_df, check_samples, fix_time_muse_hz
from pre_processing.clean_mitch_timestamps import remove_end_duplicates, add_ReconstructedTime
from pre_processing.filter_sensor_data import median_filter_medfilt
from pre_processing.segment_repetition_vol2 import plot_activity_accelerations_peaks_and_magnitude, get_start_stop_times_from_peaks, assign_rep_ids
from plotting.plot_sensor_signals import plot_all_sensors, plot_fsr, plot_single_imu
from utils import get_imu_cols, get_fsr_cols

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

### Define path to raw sensor files and output directory (same directory as test files is convention)

First, define where data files exist by modifying base_dir and participant_dir_name. Use standardized naming to make file retrieval easier! 

In [ ]:
base_dir = Path(r"C:\Users\johalot\msc_data")
participant_dir_name = "prelim_6"

data_dir = base_dir / participant_dir_name

Assuming a certain naming structure and sensor placement, run the cell below to get the data files from the participant defined in participant_dir_name.

In [ ]:
# Define path to arm sensor file (raw data)
# raw_rarm = data_dir / "Muse_E2511_RED-ARM_RIGHT2.txt"
# raw_larm = data_dir / "Muse_E2511_GREY-ARM_LEFT2.txt"
# raw_lb = data_dir / "muse_v3_3-LB2.txt"
# raw_ub = data_dir / "muse_v3-UB2.txt"
# raw_lsole = data_dir / "mitch_B0308-LEFT2.txt"
# raw_rsole = data_dir / "mitch_B0510-RIGHT2.txt"

# raw_rarm = data_dir / "right_arm_protocol_reconst_time.csv"
# raw_larm = data_dir / "left_arm_protocol_reconst_time.csv"
# raw_lb = data_dir / "lower_back_protocol_reconst_time.csv"
# raw_ub = data_dir / "upper_back_protocol_reconst_time.csv"
# raw_lsole = data_dir / "left_sole_protocol_reconst_time.csv"
# raw_rsole = data_dir / "right_sole_protocol_reconst_time.csv"

output_dir = data_dir # optionally set output dir to something else, but output being the same directory as the raw sensor files has been practice up until now :)

### Plot all data
Check data files for inconsistencies between sensors, even before time alignment.

In [ ]:
df_plot_list = make_df(muse_file_1=str(raw_rarm), muse_file_2=str(raw_larm), muse_file_3=str(raw_lb), muse_file_4=str(raw_ub), mitch_file_1=str(raw_lsole), mitch_file_2=str(raw_rsole))


#first plot from the built in timestamps (these are known to be buggy)
plot_all_sensors(
    df_plot_list, 
    time_mode='Timestamp', 
    names = ['right arm', 'left arm',  'left insole', 'right insole', 'low back', 'upper back'],
    )

# then plot by sample index (should look more aligned)
plot_all_sensors(
    df_plot_list, 
    time_mode='index', 
    names = ['right arm', 'left arm', 'left insole', 'right insole','low back', 'upper back'],
    )

### Fix time for Muse IMU sensor data

In [ ]:
df_list = make_df(muse_file_1=str(raw_rarm), muse_file_2=str(raw_larm), muse_file_3=str(raw_lb), muse_file_4=str(raw_ub)) # a list containing one element in this case...
samples, time = check_samples(df_list)
print(df_list[2])

print("Samples:", samples)
print("Time:", time)

# fixed_time_rarm = fix_time_muse_hz(df_list, 0, str(raw_rarm), 800) # 800 is the sampling frequency
# fixed_time_larm = fix_time_muse_hz(df_list, 1, str(raw_larm), 800) # 800 is the sampling frequency
# fixed_time_lb = fix_time_muse_hz(df_list, 4, str(raw_lb), 800) # 800 is the sampling frequency
# fixed_time_ub = fix_time_muse_hz(df_list, 5, str(raw_ub), 800) # 800 is the sampling frequency

# just get them for the akso files!
fixed_time_rarm = data_dir / "right_arm_protocol_reconst_time.csv"
fixed_time_larm = data_dir / "left_arm_protocol_reconst_time.csv"
fixed_time_lb = data_dir / "lower_back_protocol_reconst_time.csv"
fixed_time_ub = data_dir / "upper_back_protocol_reconst_time.csv"
time_left = data_dir / "left_sole_protocol_reconst_time.csv"
time_right = data_dir / "right_sole_protocol_reconst_time.csv"

output_dir = data_dir

# New .csv is saved to directory where the raw sensor data file resides. Path is stored in fixed_time_{whatever sensor} variables

In [ ]:
# Plot again to see time alignment between IMUs better

df_fixed_list = [
    pd.read_csv(fixed_time_rarm),
    pd.read_csv(fixed_time_larm),
    pd.read_csv(fixed_time_lb),
    pd.read_csv(fixed_time_ub)
]

plot_all_sensors(
    df_fixed_list, 
    time_mode='ReconstructedTime', 
    names = ['right arm', 'left arm', 'low back', 'upper back'],
    )


In [ ]:
# Special handle prelim 6
# firstly get the target length of df 1. crop all other dfs to this length!
larm_reconst1 = data_dir / "Muse_E2511_GREY-ARM_LEFT1_new_time_hz.csv"
larm_reconst2 = data_dir / "Muse_E2511_GREY-ARM_LEFT2_new_time_hz.csv"

# Load data
df1 = pd.read_csv(larm_reconst1)
df2 = pd.read_csv(larm_reconst2)

target_df_1_len_imu = len(df1)
print(target_df_1_len_imu)

In [ ]:
# Special handle prelim 6
# redo this for all files!
larm_reconst1 = data_dir / "Muse_V3_3-LB1_new_time_hz.csv"
larm_reconst2 = data_dir / "Muse_V3_3-LB2_new_time_hz.csv"

# Load data
df1 = pd.read_csv(larm_reconst1)
df1= df1[:target_df_1_len_imu]
df2 = pd.read_csv(larm_reconst2)

target_df_1_len = len(df1)

# Drop ALL unnamed columns immediately
df1 = df1.loc[:, ~df1.columns.str.contains("^Unnamed")]
df2 = df2.loc[:, ~df2.columns.str.contains("^Unnamed")]

# Get the last reconstructed time from df1
offset = df1["ReconstructedTime"].iloc[-1]
print(offset)



# Add offset to df2's reconstructed time column
df2["ReconstructedTime"] = df2["ReconstructedTime"] + offset
df2 = df2.iloc[1:].reset_index(drop=True) # remove first sample to avoid reconstructedtime duplicates

# Stack (concatenate) the two dataframes
merged_df = pd.concat([df1, df2], ignore_index=True)
print("DF1 duplicates:")
print(df1["ReconstructedTime"].duplicated().sum())

print("DF2 duplicates:")
print(df2["ReconstructedTime"].duplicated().sum())

print(merged_df.index)      # should be clean RangeIndex
print(merged_df.columns)   # should NOT contain Unnamed/index


# Optional: save result
merged_df.to_csv(data_dir / "merged_LB.csv", index=False)

print("Merged duplicates count:")
print(merged_df["ReconstructedTime"].duplicated().sum())

imu_time = merged_df["ReconstructedTime"].values



In [ ]:
fixed_time_larm = data_dir / "merged_ARM_LEFT.csv"
fixed_time_rarm = data_dir / "merged_ARM_RIGHT.csv"
fixed_time_lb = data_dir / "merged_LB.csv"
fixed_time_ub = data_dir / "merged_UB.csv"

### Detect spikes, init labelling and plot

Protocol was participants tapping their LEFT ARM to indicate the beginning and end of a movement label. Therefore the left arm path is used for finding labeling time marks!

In [ ]:
peak_df, peaks = detect_spikes(fixed_time_larm)
init_label_df = init_label(peak_df, peaks, str(fixed_time_larm))
#init_label_df = init_label_df.iloc[:640000] # crop a little so see plot more clearly
plot_signal_peaks(init_label_df, peaks) 

Take a look at the generated plot. See that it makes sense wrt. the number of repetitions and peaks. Sometimes some peaks will have to be filtered away. Do so below and label again if necessary. 

Note that all the cells below must be run regardless of filtering away peaks or not.


### Activity -> correct label map
Can be altered to account for irregularities in the activity set. For instance if one extra push/pull rep is done...

In [ ]:
# define rename map
# Below is the old map used for specialization project / by Maria
# rename_map = {
#     "Activity_2": "hand_up_back",
#     "Activity_4": "hands_forward",
#     "Activity_6": "hands_up",
#     "Activity_8": "push_1",
#     "Activity_9": "pull_1",
#     "Activity_10": "push_2",
#     "Activity_11": "pull_2",
#     "Activity_12": "push_3",
#     "Activity_13": "pull_3",
#     "Activity_14": "push_4",
#     "Activity_15": "pull_4",
#     "Activity_16": "push_5",
#     "Activity_17": "pull_5",
#     "Activity_19": "squatting", 
#     "Activity_21": "lifting",
#     "Activity_23": "sitting",
#     "Activity_25": "standing",
#     "Activity_27": "walking",
# }

# Rename map for preliminary data collection for M.Sc. spring 2026
rename_map = {
    "Activity_2" : "forward_lean",
    "Activity_4" : "sideways_lean",
    "Activity_6" : "torso_twist",
    "Activity_8" : "backward_lean",
    "Activity_10" : "drag_1",
    "Activity_12" : "drag_2",
    "Activity_14" : "drag_3",
    "Activity_16" : "drag_4",
    "Activity_18" : "drag_5",
    "Activity_20" : "arms_forward",
    "Activity_22" : "shoulder_load",
    "Activity_24" : "neutral_load_left",
    "Activity_26" : "neutral_load_right",
    "Activity_28" : "lying_arms_up",
    "Activity_30" : "stairs_up_1",
    "Activity_31" : "stairs_down_1",
    "Activity_32" : "stairs_up_2",
    "Activity_33" : "stairs_down_2",
    "Activity_34" : "stairs_up_3",
    "Activity_35" : "stairs_down_3",
    "Activity_36" : "stairs_up_4",
    "Activity_37" : "stairs_down_4",
    "Activity_38" : "stairs_up_5",
    "Activity_39" : "stairs_down_5",   
}

# adjusted for akso_1
# rename_map = {
#     "Activity_1" : "forward_lean",
#     "Activity_3" : "sideways_lean",
#     "Activity_5" : "torso_twist",
#     "Activity_7" : "backward_lean",
#     "Activity_9" : "drag_1",
#     "Activity_11" : "drag_2",
#     "Activity_13" : "drag_3",
#     "Activity_15" : "drag_4",
#     "Activity_17" : "drag_5",
#     "Activity_19" : "arms_forward",
#     "Activity_21" : "shoulder_load",
#     "Activity_23" : "neutral_load_left",
#     "Activity_25" : "neutral_load_right",
#     "Activity_27" : "lying_arms_up",
#     "Activity_29" : "stairs_up_1",
#     "Activity_31" : "stairs_down_1",
#     "Activity_33" : "stairs_up_2",
#     "Activity_35" : "stairs_down_2",
#     "Activity_37" : "stairs_up_3",
#     "Activity_38" : "stairs_down_3",
#     "Activity_40" : "stairs_up_4",
#     "Activity_42" : "stairs_down_4",
#     "Activity_44" : "stairs_up_5",
#     "Activity_45" : "stairs_down_5",   
# }

In [ ]:
print(peaks)

In [ ]:
init_label_df


In [ ]:


#add_peak =  86430   + 19*800

#manually_filtered_peaks = peaks
# if no peaks need be removed, otherwise:

# code for adding a peak where none was detected
manually_filtered_peaks = []
#manually_filtered_peaks = [] # potentially manually select which peaks to keep here
drop_peaks = [0,1,2] # define which peaks to drop here (0 indexed indices)

for i in range(0, len(peaks)):
    peak = peaks[i]
    # if peak == 86430:
    #     manually_filtered_peaks.append(peak)
    #     print("adding the extra peak behind", peak)
    #     manually_filtered_peaks.append(add_peak)
    
    if i == 5:
        manually_filtered_peaks.append(peak)
        print("adding the extra peak behind", peak)
        manually_filtered_peaks.append(peak + 10*800)
    elif i in drop_peaks:
        print(f"dropping peak num {i}")
        continue
    else:
        manually_filtered_peaks.append(peaks[i])



rows = []
push_idx = 1
pull_idx = 1
for i in range(len(manually_filtered_peaks) - 1):
    start_t = float(init_label_df["ReconstructedTime"].iloc[manually_filtered_peaks[i]]) 
    end_t   = float(init_label_df["ReconstructedTime"].iloc[manually_filtered_peaks[i+1]])
    # also add a 'rep_id' column to df
    init_label_df["rep_id"] = "idle"  
    try:
        if 'push' in rename_map[f"Activity_{i+1}"]:
            label = rename_map[f"Activity_{i+1}"]
            push_idx += 1
            start_t = start_t + 0.5 # filter away spikes at the beginning of push pull segments
            
        elif 'pull' in rename_map[f"Activity_{i+1}"]:
            label = rename_map[f"Activity_{i+1}"]
            pull_idx += 1
            start_t = start_t + 0.5
        else:
            # Automatically transfer labels into the rep_id column. Making already segmented activities (push, pull, drag, stairs) finished wrt. their rep ids.
            label = rename_map[f"Activity_{i+1}"]
    except Exception as e:
        print("Excepted", e)
        label = "idle"
    rows.append({
        "label": label,
        "Start Time (s)": str(start_t),
        "End Time (s)": str(end_t)
    })
    

# Sanity check: see if start stop times for each label is correct. Compare plot x-axis with the printed start stop times.
for item in rows:
    print(item)
plot_signal_peaks(init_label_df, manually_filtered_peaks)

init_label_df.head()

If everything looks good, proceed to concatenate and save...


In [ ]:
start_stop_df = pd.DataFrame(rows)

# Drop idle rows
start_stop_df = start_stop_df.drop(start_stop_df[start_stop_df['label'] == 'idle'].index)

start_stop_df.to_csv(output_dir / "start_stop_labels.csv")

# label both arm and back files
_, labeled_larm_path = apply_corrected_labels(new_time_file=str(fixed_time_larm), start_stop_csv_path=output_dir / "start_stop_labels.csv")
_, labeled_rarm_path = apply_corrected_labels(new_time_file=str(fixed_time_rarm), start_stop_csv_path=output_dir / "start_stop_labels.csv")
_, labeled_lb_path = apply_corrected_labels(new_time_file=str(fixed_time_lb), start_stop_csv_path=output_dir / "start_stop_labels.csv")
_, labeled_ub_path = apply_corrected_labels(new_time_file=str(fixed_time_ub), start_stop_csv_path=output_dir / "start_stop_labels.csv")

In [ ]:
start_stop_df

### Label Mitch data
(Also includes dropping end duplicates and adding ReconstructedTime)


Basically the exact same as Maria's run_label_mitch.py file.

In [ ]:
# SPECIAL CASE PRELIM 6


raw_lsole1 = data_dir / "mitch_B0308-LEFT1.txt"
raw_lsole2 = data_dir / "mitch_B0308-LEFT2.txt"

raw_rsole1 = data_dir / "mitch_B0510-RIGHT1.txt"
raw_rsole2 = data_dir / "mitch_B0510-RIGHT2.txt"


df_dupli_left1, outpath_dupli_left1 = remove_end_duplicates(str(raw_lsole1))
df_dupli_left2, outpath_dupli_left2 = remove_end_duplicates(str(raw_lsole2))
df_dupli_right1, outpath_dupli_right1 = remove_end_duplicates(str(raw_rsole1))
df_dupli_right2, outpath_dupli_right2 = remove_end_duplicates(str(raw_rsole2))

plt.plot(df_dupli_left1['Fsr.01'])
plt.show()

df_dupli_left1 = df_dupli_left1[:target_df_1_len_imu//8]
df_dupli_right1 = df_dupli_right1[:target_df_1_len_imu//8]

df_time_left1, outpath_time_left1 = add_ReconstructedTime(outpath_dupli_left1, df_dupli_left1)
df_time_left2, outpath_time_left2 = add_ReconstructedTime(outpath_dupli_left2, df_dupli_left2)
df_time_right1, outpath_time_right1 = add_ReconstructedTime(outpath_dupli_right1, df_dupli_right1)
df_time_right2, outpath_time_right2 = add_ReconstructedTime(outpath_dupli_right2, df_dupli_right2)

# Drop ALL unnamed columns immediately
df_time_left1 = df_time_left1.loc[:, ~df_time_left1.columns.str.contains("^Unnamed")]
df_time_left2 = df_time_left2.loc[:, ~df_time_left2.columns.str.contains("^Unnamed")]

# Get the last reconstructed time from df1
offset_left = df_time_left1["ReconstructedTime"].iloc[-1]

# Add offset to df2's reconstructed time column
df_time_left2["ReconstructedTime"] = df_time_left2["ReconstructedTime"] + offset_left
df_time_left2 = df_time_left2.iloc[1:].reset_index(drop=True) # remove first sample to avoid reconstructedtime duplicates

# Stack (concatenate) the two dataframes
merged_df_left = pd.concat([df_time_left1, df_time_left2], ignore_index=True)


df_time_right1 = df_time_right1.loc[:, ~df_time_right1.columns.str.contains("^Unnamed")]
df_time_right2 = df_time_right2.loc[:, ~df_time_right2.columns.str.contains("^Unnamed")]

# Get the last reconstructed time from df1
offset_right = df_time_right1["ReconstructedTime"].iloc[-1]
print(offset_right)

# Add offset to df2's reconstructed time column
df_time_right2["ReconstructedTime"] = df_time_right2["ReconstructedTime"] + offset_right
df_time_right2 = df_time_right2.iloc[1:].reset_index(drop=True) # remove first sample to avoid reconstructedtime duplicates

# Stack (concatenate) the two dataframes
merged_df_right = pd.concat([df_time_right1, df_time_right2], ignore_index=True)

outpath_time_left = data_dir / "merged_lsole_time.csv"
merged_df_left.to_csv(outpath_time_left)
outpath_time_right =data_dir / "merged_rsole_time.csv" 
merged_df_right.to_csv(outpath_time_right)

df_label_left, outpath_label_left = apply_corrected_labels(str(outpath_time_left), output_dir / "start_stop_labels.csv", merged_df_left)
df_label_right, outpath_label_right = apply_corrected_labels(str(outpath_time_right), output_dir / "start_stop_labels.csv", merged_df_right)

df_fsr_left, outpath_fsr_left = extract_pressure_data(outpath_label_left, df_label_left)
df_fsr_right, outpath_fsr_right = extract_pressure_data(outpath_label_right, df_label_right)




In [ ]:
plt.plot(df_fsr_left['Fsr.01'])
plt.plot(df_fsr_right['Fsr.01'])

In [ ]:
df_dupli_left, outpath_dupli_left = remove_end_duplicates(str(raw_lsole))
df_dupli_right, outpath_dupli_right = remove_end_duplicates(str(raw_rsole))

print(len(df_dupli_left))
print(len(df_dupli_right))

df_dupli_right = df_dupli_right[:len(df_dupli_left)]

plt.plot(df_dupli_right['Fsr.01'])
plt.plot(df_dupli_left['Fsr.01'])

In [ ]:
# df_dupli_left, outpath_dupli_left = remove_end_duplicates(raw_left_path)
# df_dupli_right, outpath_dupli_right = remove_end_duplicates(raw_right_path)


df_time_left, outpath_time_left = add_ReconstructedTime(outpath_dupli_left, df_dupli_left)
df_time_right, outpath_time_right = add_ReconstructedTime(outpath_dupli_right, df_dupli_right)

#df_time_left['ReconstructedTime'] = df_time_right['ReconstructedTime']

df_label_left, outpath_label_left = apply_corrected_labels(outpath_time_left, output_dir / "start_stop_labels.csv", df_time_left)
df_label_right, outpath_label_right = apply_corrected_labels(outpath_time_right, output_dir / "start_stop_labels.csv", df_time_right)

df_fsr_left, outpath_fsr_left = extract_pressure_data(outpath_label_left, df_label_left)
df_fsr_right, outpath_fsr_right = extract_pressure_data(outpath_label_right, df_label_right)

In [ ]:
# run only this for akso files
df_label_left, outpath_label_left = apply_corrected_labels(str(time_left), output_dir / "start_stop_labels.csv", None)
df_label_right, outpath_label_right = apply_corrected_labels(str(time_right), output_dir / "start_stop_labels.csv", None)

df_fsr_left, outpath_fsr_left = extract_pressure_data(outpath_label_left, df_label_left)
df_fsr_right, outpath_fsr_right = extract_pressure_data(outpath_label_right, df_label_right)

In [ ]:


#df_fsr_right['ReconstructedTime'] = df_fsr_left['ReconstructedTime'][:len(df_fsr_right)]
plt.plot(df_fsr_right['ReconstructedTime'], df_fsr_right['Fsr.01'])
plt.show()



#df_fsr_left['ReconstructedTime'] = df_fsr_right['ReconstructedTime']

plt.plot(df_fsr_left['ReconstructedTime'], df_fsr_left['Fsr.01'])


print("right reconstr. time length", len(df_fsr_right['ReconstructedTime']))
print("left reconstr. time length", len(df_fsr_left['ReconstructedTime']))

### Plot all sensors time aligned

In [ ]:
df_fixed_list = [
    pd.read_csv(fixed_time_rarm),
    pd.read_csv(fixed_time_larm),
    pd.read_csv(fixed_time_lb),
    pd.read_csv(fixed_time_ub),
    pd.read_csv(outpath_fsr_left),
    pd.read_csv(outpath_fsr_right)
]

plot_all_sensors(
    df_fixed_list, 
    time_mode='ReconstructedTime', 
    names = ['right arm', 'left arm', 'low back', 'upper back', 'left sole', 'right sole'],
    )

### Drop idle for all sensors

In [ ]:
df_left_no_idle, outpath_lsole_no_idle = remove_idle(correct_label_filepath=outpath_label_left, correct_label_df=df_label_left)
df_right_no_idle, outpath_rsole_no_idle = remove_idle(correct_label_filepath=outpath_label_right, correct_label_df=df_label_right)
df_rarm_no_idle, outpath_rarm_no_idle = remove_idle(correct_label_filepath=labeled_rarm_path)
df_larm_no_idle, outpath_larm_no_idle = remove_idle(correct_label_filepath=labeled_larm_path)
df_lb_no_idle, outpath_lb_no_idle = remove_idle(correct_label_filepath=labeled_lb_path)
df_ub_no_idle, outpath_ub_no_idle = remove_idle(correct_label_filepath=labeled_ub_path)

### Plot again

In [ ]:
df_no_idle_list = [
    df_rarm_no_idle,
    df_larm_no_idle,
    df_left_no_idle,
    df_right_no_idle,
    df_lb_no_idle,
    df_ub_no_idle
]

plot_all_sensors(
    df_no_idle_list, 
    time_mode='ReconstructedTime', 
    names = ['right arm', 'left arm', 'left insole', 'right insole','low back', 'upper back'],
    )

# Select FSR columns
fsr_cols = df_left_no_idle.filter(like="Fsr")

# Compute row-wise average
fsr_mean = fsr_cols.mean(axis=1)

# Plot vs reconstructed time
plt.plot(df_left_no_idle["ReconstructedTime"], fsr_mean)
plt.xlabel("Time")
plt.ylabel("Average FSR")
plt.title("Average FSR vs Time")
plt.show()

for df in df_no_idle_list:
    value = "neutral_load_right"

    mask = df["label"] == value
    if mask.any():
        print(df.index[mask][-1])
    print(df['ReconstructedTime'].iloc[-1])


### Apply median filter for all sensors and drop IMU columns from Mitch files

Kernel sizes: 21 for Muse sensors, 9 for mitch. Reasoning found in Maria's master's thesis, where different kernel sizes were tested.

In [ ]:
imu_columns = get_imu_cols()

fsr_columns = get_fsr_cols()

df_filtered_rarm, path_rarm_filtered = median_filter_medfilt(outpath_rarm_no_idle, imu_columns, kernel_size=21)
df_filtered_larm, path_back_larm = median_filter_medfilt(outpath_larm_no_idle, imu_columns, kernel_size=21)
df_filtered_lb, path_lb_filtered = median_filter_medfilt(outpath_lb_no_idle, imu_columns, kernel_size=21)
df_filtered_ub, path_back_ub = median_filter_medfilt(outpath_ub_no_idle, imu_columns, kernel_size=21)

df_filtered_lsole, path_lsole_filtered = median_filter_medfilt(outpath_lsole_no_idle, fsr_columns, kernel_size=9)
df_filtered_rsole, path_rsole_filtered = median_filter_medfilt(outpath_rsole_no_idle, fsr_columns, kernel_size=9)

df_filtered_lsole = df_filtered_lsole.drop(columns=imu_columns)
df_filtered_rsole = df_filtered_rsole.drop(columns=imu_columns)

df_filtered_lsole.to_csv(f"{path_lsole_filtered.rstrip('.csv')}_drop_imu.csv")
df_filtered_rsole.to_csv(f"{path_rsole_filtered.rstrip('.csv')}_drop_imu.csv")


### Find repetition IDs


##### Sanity check: same start, stop times in all reconstructed times?

In [ ]:
dfs = [df_filtered_rarm, df_filtered_larm, df_filtered_lsole, df_filtered_rsole, df_filtered_lb, df_filtered_ub]

for df in dfs:
    RT_list = df['ReconstructedTime'].to_list()
    print("Start time", RT_list[0])
    print("Stop time", RT_list[-1])
    print("--------------------------")

#### Rep id methodology:
- Extract windows of single activities
- Treat each activity one by one
- Use accel magnitude of the most relevant sensor e.g. left arm for left arm movement, lower back for back centered movements etc.
- Split so that both transients of a rep is contained within the rep.

More details in specialization project report. Alternate method in Maria's thesis.

#### Switch the activity_label argument to the desired one below and run through


In [ ]:
# we use the arm df from the previous cell
activity_label = 'lying_arms_up'
_, peaks, peaks_mask = plot_activity_accelerations_peaks_and_magnitude(df_filtered_rarm, activity_label, height=1070, distance=900)

In [ ]:
# investigate the peaks
peaks

In [ ]:
# Manually alter the peaks such that the peaks can be used to differentiate between 
drop_peaks = [14]
filtered_peaks = []
filtered_peaks_original = []
for i in range(len(peaks_mask)):
    
    # if i == 0:
    #     filtered_peaks.append(peaks_mask[i] - 0.5*800)
    #     filtered_peaks_original.append(peaks[i] - 0.5*800)
    # if i == 3:
    #     filtered_peaks.append(peaks_mask[i] - 0.5*800)
    #     filtered_peaks_original.append(peaks[i] - 0.5*800)
        

    
    # if i == 0:
    #     filtered_peaks.append(peaks_mask[i] - 0.5*800)
    #     filtered_peaks_original.append(peaks[i]- 0.5*800)
    
    
    if i not in drop_peaks:
        filtered_peaks.append(peaks_mask[i])
        filtered_peaks_original.append(peaks[i])
        
    
        
        
    
    


In [ ]:
# plot again to see that everything aligns

plot_activity_accelerations_peaks_and_magnitude(df_filtered_rarm, activity_label, peak_indices=filtered_peaks)

In [ ]:
len(filtered_peaks) == len(filtered_peaks_original)


In [ ]:
# Split in the middle between last and first peak
even = []
odd = []
even_orig = []
odd_orig = []
for i in range(len(filtered_peaks)):
    if i % 2 == 0:
        # print(f'adding index {i} to EVEN')
        # print('second mark is', df_filtered_back['ReconstructedTime'].iloc[int(filtered_peaks_original[i])])
        even.append(filtered_peaks[i])
        even_orig.append(filtered_peaks_original[i])
    else:
        # print(f'adding index {i} to ODD')
        # print('second mark is', df_filtered_back['ReconstructedTime'].iloc[int(filtered_peaks_original[i])])
        odd.append(filtered_peaks[i])
        odd_orig.append(filtered_peaks_original[i])

# print(even_orig)
# print(odd_orig)
filtered_peaks_original = [] # clear the list
for i in range(len(odd_orig)):
    first = even_orig[i]
    second = odd_orig[i]
    # print(first,second)
    # print("In time")
    # print(df_filtered_back['ReconstructedTime'].iloc[int(first)], df_filtered_back['ReconstructedTime'].iloc[int(second)],)
   # print((first + second) // 2)
    # segment at middle point between even and odd peak, i.e. last of prev rep and first of next rep:
    
    filtered_peaks_original.append((first + second) // 2) 

print("Splits seconds marks")
for idx in filtered_peaks_original:
    print(df_filtered_lb['ReconstructedTime'].iloc[int(idx)])   
    
    

#### Save the repetitions start stop times to a .csv-file
Below, the df is also visualized. Check that the timestamps correspond with the plot above:)

In [ ]:
print(filtered_peaks_original)
import importlib
import pre_processing.segment_repetition_vol2 as seg
importlib.reload(seg)
from pre_processing.segment_repetition_vol2 import get_start_stop_times_from_peaks

start_stop_reps_dict, start_stop_reps_df = start_stop_dict, intervals_df = get_start_stop_times_from_peaks(
    df=df_filtered_lb,
    peaks=filtered_peaks_original,
    activity_name=activity_label,
    num_reps=6,
    side_mode=None
)
start_stop_reps_df.to_csv(output_dir / f'start_stop_rep_times_{activity_label}.csv')


In [ ]:
# check that df makes sense
start_stop_reps_df

#### Now, add a rep_id column to each sensor dataframe based on start stop time .csv files


In [ ]:
# defien a dict with all dfs to assign rep_id for, they must have a corresponding .csv file with start_stop_times for the relevant activities. 
filtered_dfs = {
    "left_arm" : df_filtered_larm,
    "right_arm" : df_filtered_rarm,
    "upper_back" : df_filtered_ub,
    "lower_back" : df_filtered_lb,
    "left_sole" : df_filtered_lsole,
    "right_sole" : df_filtered_rsole
    }

# define a set of labels that if they occur, should not have rep_id assigned from a .csv, but rather from their label which is assumed to already be numbered after repetitions. 
numbered_labels = [
        "push",
        "pull",
        "drag",
        "stairs_up",
        "stairs_down"
]
# add a rep_id columm (just a copy of the labels to each df.)
for _, df in filtered_dfs.items():
    df["rep_id"] = df["label"].copy()
    
# run the next cell to assign rep_ids and save the final .csv files. 

In [ ]:
import pre_processing.segment_repetition_vol2 as seg
import importlib
importlib.reload(seg)
from pre_processing.segment_repetition_vol2 import assign_rep_ids

assign_rep_ids(sensor_dfs=filtered_dfs, output_dir=output_dir, numbered_labels=numbered_labels)

## Finished! 🎉
Go over output directory and delete unnecessary temp .csv files. These files might take up a lot of space if left....


## Rerun the preprocessing using the run_full_preprocessing script with the label and segmentationn files obtained here